# Capability: lookup

The `lookup` capability resolves a field value by searching known datasets (lookup tables). This is useful for mapping codes to names — for example, converting a supplier ID to a full supplier name.

In [ ]:
from pydantic import BaseModel

import paxman
import paxman.contract.adapters.pydantic


class SupplierInfo(BaseModel):
    supplier_id: str
    supplier_name: str


sample = "Supplier ID: ACME-01"

result = paxman.normalize(sample, SupplierInfo)
print(f"Data: {result.normalized_data}")

## Register a custom capability

Paxman's `register_capability` lets you add custom extraction logic. Let's build a simple supplier-code lookup.

In [ ]:
from paxman import register_capability
from paxman.capabilities.base import CapabilityContext
from paxman.capabilities.result import Candidate, CapabilityResult, EvidenceRef
from paxman.capabilities.spec import CapabilitySpec, CapabilityTier, CostHint

LOOKUP = {
    "ACME-01": "ACME Hardware Supplies",
    "GLOBEX-99": "Globex Construction",
}

_SPEC = CapabilitySpec(
    id="supplier_lookup",
    version="1.0",
    input_types=("STRING",),
    output_type="STRING",
    cost_estimate=CostHint(tokens=0, ms=1, usd=0.0),
    tier=CapabilityTier.STRUCTURED_LOOKUP,
    deterministic=True,
)


class SupplierLookup:
    @property
    def spec(self) -> CapabilitySpec:
        return _SPEC

    def invoke(self, ctx: CapabilityContext) -> CapabilityResult:
        text = ctx.raw_input.decode("utf-8", errors="replace")
        candidates: list[Candidate] = []
        for code, name in LOOKUP.items():
            if code in text:
                ev = EvidenceRef(
                    capability_id="supplier_lookup",
                    capability_version="1.0",
                    field_path=ctx.field_path,
                    context={"matched_code": code},
                )
                candidates.append(Candidate(value=name, evidence_refs=(ev,)))
        return CapabilityResult(candidates=tuple(candidates))


register_capability(SupplierLookup())
print("Registered custom supplier_lookup capability.")

In [ ]:
result2 = paxman.normalize(sample, SupplierInfo)
print(f"Data with lookup: {result2.normalized_data}")
print(f"Field results: {list(result2.field_results.keys())}")
for path, fr in result2.field_results.items():
    print(
        f"  {path}: value={fr.value!r} confidence={fr.confidence.name} status={fr.status.name}"
    )

## How it works

1. The V1 `lookup` capability is a generic framework — you provide the lookup table or function.
2. Custom capabilities are registered via `paxman.register_capability(capability_instance)`. Per ADR-0005, capabilities do **not** assign confidence — the Reconciler does.
3. The **planner** considers all registered capabilities when building the field plan.
4. **Structured-lookup** tier capabilities are preferred over tier-4 inference.

> **Reference:** See `docs/howto/add_capability.md` for the full capability authoring guide.

## Try it yourself

- Build a lookup table for product SKU to product name and register it as a capability.
- What happens if the lookup returns no candidates? Check the unresolved fields.
- Register two capabilities that match the same field — how does the reconciler choose?